# Paired Bootstrap Test and Majority Vote Analysis

This notebook implements the statistical evaluation pipeline for the prompt engineering experiment. It consolidates predictions from 3 independent evaluation rounds using majority vote, and applies the paired bootstrap test to compare prompting strategies.

The notebook is designed to run locally or in any Jupyter-compatible environment and requires environment variables defined in a `.env` file (see `.env.example` at the project root).

<br>

**Expected Environment Variables** (`.env` file)

*   `'AP_DATA_RESULTS'`: Path to results directory

<br>

**Expected project structure**

```
└── Data
   └── results
       ├── deepseek-v3.2
       ├── gemma2-9b-it
       ├── qwen3.5-9b
       ├── boto-9b-it
       └── multilingual-sentiment-analysis
```

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats as sp_stats
from sklearn.metrics import balanced_accuracy_score
from collections import Counter
from itertools import combinations
import warnings
import os
import re

from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
#Paths
RESULTS_DATA_ROOT_PATH = Path(os.getenv('AP_DATA_RESULTS'))


# Adjustments
DATASETS_TASK_TYPES = {
    "imdb-pt": "BSC",
    "opcovidbr": "BSC",
    "reli": "MSC",
    "tweetsentbr": "MSC"
    }


TASK_LABELS = {
    "BSC":[(-1, "Negative"), (1, "Positive")],
    "MSC":[(-1, "Negative"),(0, "Neutral"), (1, "Positive")]
    }


TEMPLATE_NAME_ADJUST = {
    "template-task-specific-meta-prompt-few-shot" : "FSP-AMI",
    "template-task-agnostic-no-instructions-few-shot-en" : "FSP-NI",
    "template-task-specific-instructions-few-shot" : "FSP-MTSI",
    "template-task-specific-instructions-few-shot-json" : "FSP-MTSISOF",
    "template-task-specific-instructions-zero-shot" : "ZSP-MTSI",
    "template-task-specific-roleplay-few-shot" : "FSP-MRI"
    }


TEMPLATE_SORT_ADJUST= {
    "FSP-AMI" : 5,
    "FSP-NI" : 1,
    "FSP-MTSI" : 2,
    "FSP-MTSISOF" : 3,
    "ZSP-MTSI " : 6,
    "FSP-MRI" : 4}

MODEL_NAME_ADJUSMENT = {
    "gemma2-9b-it" : "Gemma2-9B-Instruct",
    "qwen3.5-9b" : "Qwen3.5-9B",
    "multilingual-sentiment-analysis": "Weak Baseline",
    "boto-9b-it": "Boto-9B-IT",
    "deepseek-v3.2": "DeepSeek V3.2 (Strong Baseline)"
}

SEED = 42
N_BOOTSTRAP = 10_000
ALPHA = 0.05
N_RUNS = 3

COMPARISONS = [
    ("ZSP-MTSI","FSP-NI", " Instruction-only vs Demo-only"),
    ("FSP-MTSI", "FSP-NI",    "Combined vs. Demo-only"),
    ("FSP-MTSI", "ZSP-MTSI",  "Combined vs. Instruction-only"),
    ("FSP-AMI", "FSP-MTSI",    "Meta-prompt vs. Manual"),
    ("FSP-MRI", "FSP-MTSI",    "Role vs. Manual"),
    ("FSP-MTSISOF","FSP-MTSI",    "JSON-Schema vs. Manual"),
]

prompts_adj = {'template-task-specific-meta-prompt-few-shot':"Meta",
'template-task-agnostic-no-instructions-few-shot-en':"Demos-only",
'template-task-specific-instructions-few-shot':"Ist+Demos",
'template-task-specific-instructions-few-shot-json':"JSON",
'template-task-specific-instructions-zero-shot':"Inst-only",
'template-task-specific-roleplay-few-shot':"Roleplay"}

In [ ]:
def parse_mtsisof(raw: str, task_type: str) -> int:
    """
    Parser for FSP-MTSISOF: extracts the value of the 'sentimento' field
    from JSON/dict structures, ignoring wrappers (```json, <schema>, Saída:).

    Logic:
    1. Searches for the pattern 'sentimento': <value> or "sentimento": <value>
    2. Validates that <value> is a legitimate label (-1, 0, 1 for MSC; -1, 1 for BSC)
    3. Accepts optional .0 suffix (e.g., -1.0)

    Args:
        raw: Raw prediction string from the model.
        task_type: Task type ("BSC" for binary, "MSC" for multiclass).

    Returns:
        Extracted label as int, or 2 (FNC) if no valid label is found.
    """
    raw_str = str(raw)
    raw_stripped = raw_str.strip()
    if task_type == "BSC":
        # Only -1 and 1 are valid
        pattern = re.compile(r"""['"]sentimento['"]\s*:\s*(-?1)(?:\.0)?(?:\s|[},\]'"]|$)""")
    else:  # MSC
        # -1, 0, and 1 are valid
        pattern = re.compile(r"""['"]sentimento['"]\s*:\s*(-?1|0)(?:\.0)?(?:\s|[},\]'"]|$)""")

    m = pattern.search(str(raw_stripped))
    if m:
        return int(m.group(1))
    return 2  # FNC


def parse_standard(raw: str, task_type: str) -> int:
    """
    Parser for all other formats (FSP-NI, FSP-MTSI, FSP-MRI, FSP-AMI, ZSP-MTSI).

    Three-layer strategy with priority:

    Layer 1 (Strict): Label at the START of the response, with optional "A:" prefix.
        Accepts: "1", "-1", "A: 0", "A:1", "A:-1.0", "A:1 \n\n**Justificativa"
        Rejects: "Etapa 1:", "A resposta é 1", "Vamos analisar..."

    Layer 2 (Moderate): Pattern "A resposta...é X" where X is the label.
        Accepts: "A resposta é 1.", "A resposta para a pergunta é: -1"
        Rejects: "A resposta para a pergunta \"sandy...\" (quotes block)

    Layer 3 (Moderate): Pattern "A classificação...é X".
        Accepts: "A classificação para o texto fornecido é 0"
        Rejects: "A classificação do sentimento para o texto forne" (no label)

    Args:
        raw: Raw prediction string from the model.
        task_type: Task type ("BSC" for binary, "MSC" for multiclass).

    Returns:
        Extracted label as int, or 2 (FNC) if no valid label is found.
    """
    raw_str = str(raw)
    raw_stripped = raw_str.strip()

    if task_type == "BSC":
        lp = r'(-?1)'      # label pattern: only -1 and 1
    else:
        lp = r'(-?1|0)'    # label pattern: -1, 0, 1

    # Layer 1: Label at start (with optional "A:", optional ".0")
    # ^ anchors to start of string (no MULTILINE)
    # (?:A\s*:?\s*)? — optional "A:" prefix with flexible spaces
    # {lp}(?:\.0)? — the label with optional .0 suffix
    # \b — word boundary to avoid capturing "1984", "1/10", etc.
    p1 = re.compile(rf'^\s*(?:A\s*:?\s*)?{lp}(?:\.0)?\b')
    m = p1.match(raw_stripped)
    if m:
        return int(m.group(1))

    # Layer 2: "A resposta...é [:]  [**] X"
    # [^"]*? — non-greedy, stops before any quote (blocks "A resposta para a pergunta \"texto...\"")
    # \bé\s*:?\s*\**\s* — "é" followed by optional ":" and optional "**"
    # trailing match — accepts period, space, asterisks, or end of string
    p2 = re.compile(rf'^A resposta[^"]*?\bé\s*:?\s*\**\s*{lp}(?:\.0)?(?:\**|[.\s]|$)')
    m = p2.match(raw_stripped)
    if m:
        return int(m.group(1))

    # Layer 3: "A classificação...é X" (same pattern as Layer 2)
    p3 = re.compile(rf'^A classificação[^"]*?\bé\s*:?\s*\**\s*{lp}(?:\.0)?(?:\**|[.\s]|$)')
    m = p3.match(raw_stripped)
    if m:
        return int(m.group(1))

    return 2  # FNC


In [ ]:
RESULT_DATASETS = {}

for csv_file in RESULTS_DATA_ROOT_PATH.glob('*/*.csv'):
    if "multilingual-sentiment-analysis" in str(csv_file):
        file_name = csv_file.stem
        model_name = file_name.split("_")[0]
        dataset_name = file_name.split("_")[1]
        template_name = None
        round = None
        task = DATASETS_TASK_TYPES[dataset_name]

    else:
        file_name = csv_file.stem
        model_name = file_name.split("_")[0]
        dataset_name = file_name.split("_")[1]
        template_name = file_name.split("_")[2]
        round = file_name.split("_")[3]
        task = DATASETS_TASK_TYPES[dataset_name]

    if model_name not in RESULT_DATASETS:
        RESULT_DATASETS[model_name] = {}

    if dataset_name not in RESULT_DATASETS[model_name]:
        RESULT_DATASETS[model_name][dataset_name] = {}

    if template_name not in RESULT_DATASETS[model_name][dataset_name]:
        RESULT_DATASETS[model_name][dataset_name][template_name] = {}

    if round not in RESULT_DATASETS[model_name][dataset_name][template_name]:
        RESULT_DATASETS[model_name][dataset_name][template_name][round] = pd.read_csv(csv_file)
        RESULT_DATASETS[model_name][dataset_name][template_name][round]["round"] = round

        RESULT_DATASETS[model_name][dataset_name][template_name][round]["predictions_adj"] = RESULT_DATASETS[model_name][dataset_name][template_name][round]["predictions"].apply(
            lambda x: parse_mtsisof(x, task) if template_name=="template-task-specific-instructions-few-shot-json"
            else parse_standard(x, task))

        RESULT_DATASETS[model_name][dataset_name][template_name][round]["ac.b."] = balanced_accuracy_score(
            y_true=RESULT_DATASETS[model_name][dataset_name][template_name][round]["label"],
            y_pred=RESULT_DATASETS[model_name][dataset_name][template_name][round]["predictions_adj"])

In [ ]:
df_list = []

for model in RESULT_DATASETS.keys():
    for dataset_name in RESULT_DATASETS[model].keys():
        for prompt in RESULT_DATASETS[model][dataset_name].keys():
            for run in RESULT_DATASETS[model][dataset_name][prompt].keys():
                temp_df = RESULT_DATASETS[model][dataset_name][prompt][run]
                temp_df["model"] = model
                temp_df["dataset"] = dataset_name
                temp_df["prompt"] = prompt
                temp_df["run"] = run
                df_list.append(temp_df)

final_df = pd.concat(df_list)
final_df.reset_index(drop=True, inplace=True)
final_df.info()

In [ ]:
def load_predictions(model: str, dataset: str, prompt: str, run: int) -> tuple[np.ndarray, np.ndarray]:
    """
    Load ground truth labels and adjusted predictions for a given model/dataset/prompt/run.

    Args:
        model: Model identifier string.
        dataset: Dataset name.
        prompt: Prompt template name (or None for multilingual-sentiment-analysis).
        run: Evaluation round number (1-indexed).

    Returns:
        Tuple of (y_true, y_pred) as numpy arrays.
    """
    if model != "multilingual-sentiment-analysis":
        return (
            RESULT_DATASETS[model][dataset][prompt][str(run)]["label"],
            RESULT_DATASETS[model][dataset][prompt][str(run)]["predictions_adj"]
                )
    else:
        return (
            RESULT_DATASETS[model][dataset][None][None]["label"],
            RESULT_DATASETS[model][dataset][None][None]["predictions_adj"]
                )


In [ ]:
def majority_vote(predictions_runs: list[np.ndarray],
                  fnc_code: int = 2) -> np.ndarray:
    """
    Consolidates predictions from multiple runs via majority vote.

    For each instance i, selects the most frequent label among the K runs.
    In case of a tripartite tie (K=3, MSC: each run predicts a different class),
    assigns fnc_code (non-conforming format), following the conservative principle
    that a prediction without convergence is treated as unusable.

    For BSC with K=3 (odd, 2 classes), ties are impossible.

    Args:
        predictions_runs: List of K arrays, each with N predictions.
        fnc_code: Code assigned to unresolvable ties (default=2).

    Returns:
        np.ndarray of shape (N,) — consolidated prediction per instance.
    """
    n_instances = len(predictions_runs[0])
    n_runs = len(predictions_runs)
    consolidated = np.zeros(n_instances, dtype=predictions_runs[0].dtype)

    for i in range(n_instances):
        votes = [run_preds[i] for run_preds in predictions_runs]
        counter = Counter(votes)
        top_label, top_count = counter.most_common(1)[0]

        if top_count > n_runs / 2:
            # Strict majority: at least 2/3 agree
            consolidated[i] = top_label
        else:
            # Tripartite tie: no class has majority
            consolidated[i] = fnc_code

    return consolidated


def compute_agreement_stats(predictions_runs: list[np.ndarray],
                            fnc_code: int = 2) -> dict:
    """
    Computes agreement statistics across runs, including the proportion of
    tripartite ties (relevant for MSC with K=3).

    Categories:
    - unanimous:  all K runs predict the same class
    - majority:   strict majority (>K/2), but not unanimous
    - tripartite: all predictions are distinct (no majority)

    Args:
        predictions_runs: List of K arrays, each with N predictions.
        fnc_code: Code for non-conforming predictions (default=2).

    Returns:
        Dictionary with agreement statistics.
    """
    n_instances = len(predictions_runs[0])
    n_runs = len(predictions_runs)

    unanimous = 0
    majority = 0
    tripartite = 0

    for i in range(n_instances):
        votes = [run[i] for run in predictions_runs]
        n_distinct = len(set(votes))
        top_count = Counter(votes).most_common(1)[0][1]

        if n_distinct == 1:
            unanimous += 1
        elif top_count > n_runs / 2:
            majority += 1
        else:
            tripartite += 1

    return {
        "n_instances": n_instances,
        "n_runs": n_runs,
        "unanimous": unanimous,
        "unanimous_pct": unanimous / n_instances * 100,
        "majority": majority,
        "majority_pct": majority / n_instances * 100,
        "tripartite_ties": tripartite,
        "tripartite_ties_pct": tripartite / n_instances * 100,
        "convergent": unanimous + majority,
        "convergent_pct": (unanimous + majority) / n_instances * 100,
    }


In [ ]:
def paired_bootstrap_test(
    y_true: np.ndarray,
    preds_a: np.ndarray,
    preds_b: np.ndarray,
    n_bootstrap: int = 10_000,
    alpha: float = 0.05,
    seed: int = 42,
) -> dict:
    """
    Bilateral paired bootstrap test to compare two systems.

    Operates on consolidated predictions (post majority vote):
    one prediction per instance x prompt. Pairing is by instance
    (same instance evaluated by both systems).

    Procedure:
    1. Compute delta_obs = BAcc(A) - BAcc(B) on the original test set
    2. For b = 1..B:
       a. Sample N indices with replacement
       b. Compute delta*_b = BAcc(A[idx]) - BAcc(B[idx])
    3. Two-sided p-value: P(|delta*_centered| >= |delta_obs|)
    4. CI: percentiles alpha/2 and 1-alpha/2 of the delta* distribution

    Args:
        y_true: Ground truth labels.
        preds_a: Predictions from system A.
        preds_b: Predictions from system B.
        n_bootstrap: Number of bootstrap samples.
        alpha: Significance level.
        seed: Random seed for reproducibility.

    Returns:
        dict with: bacc_a, bacc_b, delta, p_value, ci_lower, ci_upper,
                   significant, bootstrap_deltas, fnc_rate_a, fnc_rate_b,
                   fnc_rate_delta.
    """
    rng = np.random.RandomState(seed)
    n = len(y_true)

    # Validations
    assert len(preds_a) == n, f"preds_a has {len(preds_a)} elements, expected {n}"
    assert len(preds_b) == n, f"preds_b has {len(preds_b)} elements, expected {n}"

    # Observed metrics
    bacc_a = balanced_accuracy_score(y_true, preds_a)
    bacc_b = balanced_accuracy_score(y_true, preds_b)
    observed_delta = bacc_a - bacc_b

    fnc_rate_a = np.mean(preds_a == 2)
    fnc_rate_b = np.mean(preds_b == 2)
    fnc_rate_observed_delta = fnc_rate_a - fnc_rate_b

    # Bootstrap resampling
    deltas = np.zeros(n_bootstrap)
    fnc_rate_deltas = np.zeros(n_bootstrap)
    for i in range(n_bootstrap):
        idx = rng.randint(0, n, size=n)
        boot_true = y_true[idx]
        boot_a = preds_a[idx]
        boot_b = preds_b[idx]

        # Guard against degenerate resamples
        # (all instances from same class — rare but possible in small datasets)
        if len(np.unique(boot_true)) < 2:
            deltas[i] = 0.0
            continue

        deltas[i] = (
            balanced_accuracy_score(boot_true, boot_a)
            - balanced_accuracy_score(boot_true, boot_b)
        )

    # Two-sided p-value (centered under H0: delta = 0)
    centered_deltas = deltas - deltas.mean()
    p_value = np.mean(np.abs(centered_deltas) >= np.abs(observed_delta))

    # Percentile confidence interval for BAcc
    ci_lower = np.percentile(deltas, 100 * alpha / 2)
    ci_upper = np.percentile(deltas, 100 * (1 - alpha / 2))

    return {
        "bacc_a": bacc_a,
        "bacc_b": bacc_b,
        "delta": observed_delta,
        "p_value": p_value,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "significant": p_value < alpha,
        "bootstrap_deltas": deltas,
        "fnc_rate_a": fnc_rate_a,
        "fnc_rate_b": fnc_rate_b,
        "fnc_rate_delta": fnc_rate_observed_delta,
    }


In [ ]:
def consolidate_predictions(
    model: str,
    prompt: str,
    dataset: str,
) -> tuple[np.ndarray, np.ndarray, dict]:
    """
    Consolidation pipeline for a (model x prompt x dataset) triple:
    1. Loads predictions from 3 runs
    2. Verifies consistency of y_true across runs
    3. Applies majority vote
    4. Computes agreement statistics

    Args:
        model: Model identifier string.
        prompt: Prompt template name.
        dataset: Dataset name.

    Returns:
        y_true: Ground truth labels as np.ndarray.
        y_pred_voted: Consolidated predictions via majority vote as np.ndarray.
        stats: dict with agreement statistics and BAcc per run.
    """
    all_preds = []
    all_ytrue = []
    per_run_bacc = []
    per_run_fnc_rate = []

    for run in range(1, N_RUNS + 1):
        y_true, y_pred = load_predictions(model, dataset, prompt, run)
        all_preds.append(y_pred)
        all_ytrue.append(y_true)
        per_run_bacc.append(balanced_accuracy_score(y_true, y_pred))
        per_run_fnc_rate.append(np.mean(y_pred == 2)*100)


    # Verify y_true is identical across all runs
    for run_idx in range(1, N_RUNS):
        assert np.array_equal(all_ytrue[0], all_ytrue[run_idx]), (
            f"ERROR: y_true differs between run 1 and run {run_idx + 1} "
            f"for {model}/{prompt}/{dataset}. "
            f"Verify that instances are in the same order."
        )

    y_true = all_ytrue[0]

    # Majority vote
    y_pred_voted = majority_vote(all_preds)

    # Agreement statistics
    agreement = compute_agreement_stats(all_preds)

    # BAcc of majority vote
    bacc_voted = balanced_accuracy_score(y_true, y_pred_voted)
    fnc_voted = np.mean(y_pred_voted == 2)


    stats = {
        **agreement,
        "per_run_bacc": per_run_bacc,
        "per_run_fnc" : per_run_fnc_rate,
        "mean_bacc": np.mean(per_run_bacc),
        "std_bacc": np.std(per_run_bacc, ddof=1) if N_RUNS > 1 else 0.0,
        "cv": (np.std(per_run_bacc, ddof=1) / np.mean(per_run_bacc)
               if np.mean(per_run_bacc) > 0 and N_RUNS > 1 else 0.0),
        "fnc_rate": fnc_voted*100,
        "bacc_majority_vote": bacc_voted,
    }

    return y_true, y_pred_voted, stats


In [ ]:
def print_summary(df: pd.DataFrame) -> None:
    """Prints a general summary of bootstrap test results."""

    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)

    df = df[~df["model"].isin(["multilingual-sentiment-analysis","deepseek-v3.2"])]

    total = len(df)
    sig = df["significant"].sum()
    ns = total - sig

    print(f"\n  Total comparisons: {total}")
    print(f"  Significant (p < {ALPHA}): {sig} ({sig/total*100:.1f}%)")
    print(f"  Non-significant: {ns} ({ns/total*100:.1f}%)")

    # By comparison type
    print(f"\n  By comparison type:")
    for label in df["comparison"].unique():
        subset = df[df["comparison"] == label]
        s = subset["significant"].sum()
        t = len(subset)
        mean_delta = subset["delta"].mean()
        print(f"    {label:<35s}: {s}/{t} sig.  (mean delta = {mean_delta:+.4f})")

    # By task
    print(f"\n  By task:")
    for task in ["BSC", "MSC"]:
        subset = df[df["task"] == task]
        s = subset["significant"].sum()
        t = len(subset)
        print(f"    {task}: {s}/{t} significant")

    # By model
    print(f"\n  By model:")
    for model in MODEL_NAME_ADJUSMENT.keys():
        subset = df[df["model"] == model]
        s = subset["significant"].sum()
        t = len(subset)
        print(f"    {model:25s}: {s}/{t} significant")

    # Central hypothesis: combined formats vs. isolated
    print(f"\n  Central hypothesis (combined > isolated):")
    for label in ["Combined vs. Demo-only", "Combined vs. Instruction-only"]:
        subset = df[df["comparison"] == label]
        s = subset["significant"].sum()
        t = len(subset)
        mean_p = subset["p_value"].mean()
        all_positive = (subset["delta"] > 0).all()
        direction = "A always > B" if all_positive else "mixed direction"
        print(f"    {label:<35s}: {s}/{t} sig., mean p={mean_p:.4f}, {direction}, mean delta = {subset['delta'].mean():+.4f}")


In [ ]:
def run_full_analysis() -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Full analysis pipeline:
    1. Consolidate predictions via majority vote
    2. Run paired bootstrap for each comparison of interest
    3. Generate result tables

    Returns:
        Tuple of (df_consolidation, df_bootstrap) DataFrames.
    """

    os.makedirs(RESULTS_DATA_ROOT_PATH, exist_ok=True)

    # ─── PHASE 1: Consolidation ───
    print("=" * 70)
    print("PHASE 1: Majority Vote Consolidation")
    print("=" * 70)

    consolidated = {}  # key: (model, prompt, dataset) -> (y_true, y_pred_voted)
    consolidation_stats = []

    for model in MODEL_NAME_ADJUSMENT.keys():
        for dataset_name in DATASETS_TASK_TYPES.keys():
            for prompt in TEMPLATE_NAME_ADJUST.keys():
                y_true, y_pred_voted, stats = consolidate_predictions(
                    model, prompt, dataset_name
                )
                consolidated[(model, prompt, dataset_name)] = (y_true, y_pred_voted)

                consolidation_stats.append({
                    "model": model,
                    "dataset": dataset_name,
                    "prompt": prompt,
                    "task": DATASETS_TASK_TYPES[dataset_name],
                    "mean_bacc_runs": stats["mean_bacc"],
                    "std_bacc_runs": stats["std_bacc"],
                    "cv": stats["cv"],
                    "bacc_majority_vote": stats["bacc_majority_vote"],
                    "fnc_majority_vote": stats["fnc_rate"],
                    "unanimous_pct": stats["unanimous_pct"],
                    "majority_pct": stats["majority_pct"],
                    "run1_bacc": stats["per_run_bacc"][0],
                    "run2_bacc": stats["per_run_bacc"][1],
                    "run3_bacc": stats["per_run_bacc"][2],
                    "run1_fnc": stats["per_run_fnc"][0],
                    "run2_fnc": stats["per_run_fnc"][1],
                    "run3_fnc": stats["per_run_fnc"][2],
                })

                print(
                    f"  {model:25s} | {dataset_name:12s} | {TEMPLATE_NAME_ADJUST[prompt]:12s} | "
                    f"BAcc(MV)={stats['bacc_majority_vote']:.4f}  "
                    f"Mean={stats['mean_bacc']:.4f}+/-{stats['std_bacc']:.4f}  "
                    f"Unan={stats['unanimous_pct']:.1f}%"
                )

    df_consolidation = pd.DataFrame(consolidation_stats)
    df_consolidation["prompt"] = df_consolidation["prompt"].map(TEMPLATE_NAME_ADJUST)

    df_consolidation.to_csv(
        os.path.join(RESULTS_DATA_ROOT_PATH, "consolidation_stats.csv"), index=False
    )
    print(f"\n  Saved: {RESULTS_DATA_ROOT_PATH}/consolidation_stats.csv")

    # ─── PHASE 2: Paired Bootstrap ───
    print("\n" + "=" * 70)
    print("PHASE 2: Paired Bootstrap Test (on consolidated predictions)")
    print(f"  Configuration: B={N_BOOTSTRAP}, alpha={ALPHA}, seed={SEED}")
    print("=" * 70)

    bootstrap_results = []

    for model in MODEL_NAME_ADJUSMENT.keys():
        for dataset_name in DATASETS_TASK_TYPES.keys():
            task = DATASETS_TASK_TYPES[dataset_name]

            print(f"\n{'─' * 70}")
            print(f"  {model} x {dataset_name} ({task})")
            print(f"{'─' * 70}")
            print(f"  {'Comparison':<35s} {'BAcc_A':>7s} {'BAcc_B':>7s} "
                  f"{'Delta':>8s} {'p-value':>8s} {'CI 95%':>20s} {'Sig':>5s}")
            print(f"  {'─' * 90}")

            for prompt_a, prompt_b, label in COMPARISONS:
                prompt_a = list(TEMPLATE_NAME_ADJUST.keys())[list(TEMPLATE_NAME_ADJUST.values()).index(prompt_a)]
                prompt_b = list(TEMPLATE_NAME_ADJUST.keys())[list(TEMPLATE_NAME_ADJUST.values()).index(prompt_b)]

                y_true_a, preds_a = consolidated[(model, prompt_a, dataset_name)]
                y_true_b, preds_b = consolidated[(model, prompt_b, dataset_name)]

                # Pairing: y_true must be identical
                assert np.array_equal(y_true_a, y_true_b), (
                    f"y_true diverges between {prompt_a} and {prompt_b} in {dataset_name}"
                )

                result = paired_bootstrap_test(
                    y_true=y_true_a,
                    preds_a=preds_a,
                    preds_b=preds_b,
                    n_bootstrap=N_BOOTSTRAP,
                    alpha=ALPHA,
                    seed=SEED,
                )

                sig_marker = "***" if result["significant"] else "n.s."
                ci_str = f"[{result['ci_lower']:+.4f}, {result['ci_upper']:+.4f}]"

                print(
                    f"  {label:<35s} {result['bacc_a']:>7.4f} {result['bacc_b']:>7.4f} "
                    f"{result['delta']:>+8.4f} {result['p_value']:>8.4f} "
                    f"{ci_str:>20s} {sig_marker:>5s}"
                    f"{result['fnc_rate_delta']:+.4%}"
                )

                bootstrap_results.append({
                    "model": model,
                    "dataset": dataset_name,
                    "task": task,
                    "prompt_a": prompt_a,
                    "prompt_b": prompt_b,
                    "comparison": label,
                    "bacc_a": result["bacc_a"],
                    "bacc_b": result["bacc_b"],
                    "delta": result["delta"],
                    "p_value": result["p_value"],
                    "ci_lower": result["ci_lower"],
                    "ci_upper": result["ci_upper"],
                    "significant": result["significant"],
                    "fnc_rate_a": result["fnc_rate_a"],
                    "fnc_rate_b": result["fnc_rate_b"],
                    "fnc_rate_observed_delta": result["fnc_rate_delta"],
                })

    df_bootstrap = pd.DataFrame(bootstrap_results)
    df_bootstrap.to_csv(
        os.path.join(RESULTS_DATA_ROOT_PATH, "bootstrap_results.csv"), index=False
    )
    print(f"\n  Saved: {RESULTS_DATA_ROOT_PATH}/bootstrap_results.csv")

    # ─── PHASE 3: Summaries and tables ───
    print_summary(df_bootstrap)

    return df_consolidation, df_bootstrap


In [ ]:
print("=" * 70)
print("  PAIRED BOOTSTRAP TEST")
print("  Majority Vote (3 runs) + Paired Bootstrap")
print("=" * 70)
print()

df_consolidation, df_bootstrap = run_full_analysis()

print("\n" + "=" * 70)
print("  EXECUTION COMPLETE")
print(f"  Files saved to: {RESULTS_DATA_ROOT_PATH}/")
print(f"    - consolidation_stats.csv")
print(f"    - bootstrap_results.csv")
print("=" * 70)
